In [1]:
from IPython.display import Image, display
import pandas as pd

### yellowish mask
![text](yellowish_mask_preview.png)

### Test (labels)
![text](m05-label.png)

### Train (no labels)
![text](m05-nolabel.png)

In [2]:
img_label = 'm05-label.png'
img_nolabel = 'm05-nolabel.png'

test_path = 'test/'
train_path = 'train/'

In [16]:
import cv2
import numpy as np
from pathlib import Path

img_label = 'm05-label.png'
img_nolabel = 'm05-nolabel.png'

train_path = Path('train'); train_path.mkdir(exist_ok=True)
test_path  = Path('test');  test_path.mkdir(exist_ok=True)

TILE = 256

# Paste your sampled hexes here (examples only!)
TARGET_HEXES = [
    "#FDFD01",
    "#95951E",
    "#D8D80C",
    "#E4E407",
    "#BFBF15"
]

# How close a pixel must be to one of the target colors to count as "yellow"
# Start with 35–60; increase if you miss parts of the line, decrease if false positives.
TOL = 98  # Euclidean distance in RGB space

# How many matched pixels in a tile => yellow1
MIN_MATCH_PIXELS = 20  # start low for thin lines (5–30)

def read_as_bgr(path):
    im = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if im is None:
        raise FileNotFoundError(path)
    if im.ndim == 2:
        im = cv2.cvtColor(im, cv2.COLOR_GRAY2BGR)
    elif im.ndim == 3 and im.shape[2] == 4:
        im = cv2.cvtColor(im, cv2.COLOR_BGRA2BGR)
    return im

def hex_to_bgr(hex_str):
    hex_str = hex_str.lstrip('#')
    r = int(hex_str[0:2], 16)
    g = int(hex_str[2:4], 16)
    b = int(hex_str[4:6], 16)
    return np.array([b, g, r], dtype=np.int16)  # BGR

label_img = read_as_bgr(img_label)
clean_img = read_as_bgr(img_nolabel)

# Crop to common size if needed
H = min(label_img.shape[0], clean_img.shape[0])
W = min(label_img.shape[1], clean_img.shape[1])
label_img = label_img[:H, :W]
clean_img = clean_img[:H, :W]

targets_bgr = np.stack([hex_to_bgr(h) for h in TARGET_HEXES], axis=0)  # (K,3)

# Build a per-pixel mask: pixel matches if it's within TOL of ANY target color
# Vectorized: compute distance to each target, then take min.
img_i16 = label_img.astype(np.int16)  # (H,W,3)

# dist^2 to each target: (H,W,K)
diff = img_i16[:, :, None, :] - targets_bgr[None, None, :, :]
dist2 = np.sum(diff * diff, axis=3)
min_dist = np.sqrt(np.min(dist2, axis=2))

yellow_mask = (min_dist <= TOL).astype(np.uint8) * 255

# Light denoise (optional)
kernel = np.ones((3,3), np.uint8)
yellow_mask = cv2.morphologyEx(yellow_mask, cv2.MORPH_OPEN, kernel, iterations=1)

cv2.imwrite("yellow_mask_rgb.png", yellow_mask)
print("Wrote yellow_mask_rgb.png (white = detected yellow pixels)")

base = Path(img_nolabel).stem.replace('-nolabel', '')

yellow1 = 0
yellow0 = 0
tiles = 0

for y in range(0, H - TILE + 1, TILE):
    for x in range(0, W - TILE + 1, TILE):
        label_tile = label_img[y:y+TILE, x:x+TILE]
        clean_tile = clean_img[y:y+TILE, x:x+TILE]
        tile_mask  = yellow_mask[y:y+TILE, x:x+TILE]

        match_pixels = int((tile_mask > 0).sum())
        has_yellow = int(match_pixels >= MIN_MATCH_PIXELS)

        if has_yellow: yellow1 += 1
        else: yellow0 += 1

        fname = f"{base}_x{x:05d}_y{y:05d}_yellow{has_yellow}.png"
        cv2.imwrite(str(train_path / fname), label_tile)  # labeled tiles
        cv2.imwrite(str(test_path  / fname), clean_tile)  # clean tiles
        tiles += 1

print(f"Saved {tiles} tiles to train/ and test/")
print(f"Counts: yellow1={yellow1}, yellow0={yellow0}")


/var/folders/cf/4zrkk0lx1nbdx9tpklpk4jc00000gn/T/ipykernel_2219/488506430.py:64: RuntimeWarning: invalid value encountered in sqrt
  min_dist = np.sqrt(np.min(dist2, axis=2))


Wrote yellow_mask_rgb.png (white = detected yellow pixels)
Saved 432 tiles to train/ and test/
Counts: yellow1=373, yellow0=59


In [16]:
import cv2
import numpy as np
from pathlib import Path

img_label = 'm05-label.png'
img_nolabel = 'm05-nolabel.png'

train_path = Path('train'); train_path.mkdir(exist_ok=True)
test_path = Path('test'); test_path.mkdir(exist_ok=True)

TILE = 256
MIN_YELLOW_PIXELS = 40     # tile-level threshold
DELTA_E_THR = 18           # color distance tolerance (try 12–25)

def read_as_bgr(path):
    im = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if im is None:
        raise FileNotFoundError(path)
    if im.ndim == 2:
        im = cv2.cvtColor(im, cv2.COLOR_GRAY2BGR)
    elif im.shape[2] == 4:
        im = cv2.cvtColor(im, cv2.COLOR_BGRA2BGR)
    return im

label_img = read_as_bgr(img_label)
clean_img = read_as_bgr(img_nolabel)

# crop to common size if needed
H = min(label_img.shape[0], clean_img.shape[0])
W = min(label_img.shape[1], clean_img.shape[1])
label_img = label_img[:H, :W]
clean_img = clean_img[:H, :W]

# --- 1) Broad "yellow-ish" candidate mask (just to collect samples) ---
b, g, r = cv2.split(label_img)
cand = (r > 140) & (g > 140) & (b < 170) & ((r + g) > (b + 180))

cand_idx = np.where(cand)
if cand_idx[0].size < 50:
    raise ValueError(
        "Couldn't find enough yellow-ish pixels to learn color. "
        "Your annotation may not be yellow, or thresholds too strict."
    )

# --- 2) Learn the dominant annotation color from candidates ---
# sample up to N pixels for speed
N = 50000
coords = np.column_stack(cand_idx)
if coords.shape[0] > N:
    coords = coords[np.random.choice(coords.shape[0], N, replace=False)]

sample_bgr = label_img[coords[:,0], coords[:,1], :].astype(np.uint8)

# Convert samples to Lab; use median as robust "typical yellow"
sample_lab = cv2.cvtColor(sample_bgr.reshape(-1, 1, 3), cv2.COLOR_BGR2LAB).reshape(-1, 3)
target_lab = np.median(sample_lab, axis=0).astype(np.float32)
print("Learned annotation Lab (median):", target_lab)

# --- 3) Create a tight mask via Lab distance (ΔE-ish simple Euclidean) ---
label_lab = cv2.cvtColor(label_img, cv2.COLOR_BGR2LAB).astype(np.float32)
dist = np.linalg.norm(label_lab - target_lab, axis=2)
yellow_mask = dist <= DELTA_E_THR

# clean up thin noise
yellow_u8 = (yellow_mask.astype(np.uint8) * 255)
kernel = np.ones((3,3), np.uint8)
yellow_u8 = cv2.morphologyEx(yellow_u8, cv2.MORPH_OPEN, kernel, iterations=1)
yellow_u8 = cv2.morphologyEx(yellow_u8, cv2.MORPH_DILATE, kernel, iterations=1)

# optional: write out mask to visually confirm
cv2.imwrite("yellow_detected_mask.png", yellow_u8)
print("Wrote yellow_detected_mask.png (white = detected yellow)")

base = Path(img_nolabel).stem.replace('-nolabel', '')

yellow1_count = 0
yellow0_count = 0
tile_count = 0

for y in range(0, H - TILE + 1, TILE):
    for x in range(0, W - TILE + 1, TILE):
        label_tile = label_img[y:y+TILE, x:x+TILE]
        clean_tile = clean_img[y:y+TILE, x:x+TILE]
        tile_mask = yellow_u8[y:y+TILE, x:x+TILE]

        yellow_pixels = int((tile_mask > 0).sum())
        has_yellow = int(yellow_pixels >= MIN_YELLOW_PIXELS)

        if has_yellow: yellow1_count += 1
        else: yellow0_count += 1

        fname = f"{base}_x{x:05d}_y{y:05d}_yellow{has_yellow}.png"
        cv2.imwrite(str(train_path / fname), label_tile)
        cv2.imwrite(str(test_path / fname), clean_tile)
        tile_count += 1

print(f"Saved {tile_count} tiles to train/ and test/")
print(f"Label counts: yellow1={yellow1_count}, yellow0={yellow0_count}")


Learned annotation Lab (median): [231. 108. 217.]
Wrote yellow_detected_mask.png (white = detected yellow)
Saved 432 tiles to train/ and test/
Label counts: yellow1=0, yellow0=432


In [14]:
import cv2
import numpy as np
from pathlib import Path

img_label = 'm05-label.png'
img_nolabel = 'm05-nolabel.png'

train_path = Path('train'); train_path.mkdir(exist_ok=True)
test_path = Path('test'); test_path.mkdir(exist_ok=True)

TILE = 256
DIFF_THR = 25
MIN_OVERLAY_PIXELS = 40

def read_as_bgr(path: str):
    im = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if im is None:
        raise FileNotFoundError(path)

    # Normalize to 3-channel BGR
    if im.ndim == 2:  # grayscale
        im = cv2.cvtColor(im, cv2.COLOR_GRAY2BGR)
    elif im.ndim == 3 and im.shape[2] == 4:  # BGRA
        im = cv2.cvtColor(im, cv2.COLOR_BGRA2BGR)
    elif im.ndim == 3 and im.shape[2] == 3:
        pass
    else:
        raise ValueError(f"Unexpected shape for {path}: {im.shape}")

    return im

label_img = read_as_bgr(img_label)
clean_img = read_as_bgr(img_nolabel)

print("Loaded shapes:", label_img.shape, clean_img.shape)

# Crop both to common size (top-left) so absdiff works
H = min(label_img.shape[0], clean_img.shape[0])
W = min(label_img.shape[1], clean_img.shape[1])
label_img = label_img[:H, :W]
clean_img = clean_img[:H, :W]

h, w = label_img.shape[:2]

# ---- Build overlay mask via difference ----
diff = cv2.absdiff(label_img, clean_img)
diff_gray = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)
changed = diff_gray >= DIFF_THR

# Optional yellow-ish confirmation on labeled image
b, g, r = cv2.split(label_img)
yellowish = (r >= 120) & (g >= 120) & (b <= 140) & ((r + g) > (b + 120))

overlay_mask = changed & yellowish

overlay_u8 = (overlay_mask.astype(np.uint8) * 255)
kernel = np.ones((3, 3), np.uint8)
overlay_u8 = cv2.morphologyEx(overlay_u8, cv2.MORPH_OPEN, kernel, iterations=1)
overlay_u8 = cv2.morphologyEx(overlay_u8, cv2.MORPH_DILATE, kernel, iterations=1)

base = Path(img_nolabel).stem.replace('-nolabel', '')

yellow1_count = 0
yellow0_count = 0
tile_count = 0

for y in range(0, h - TILE + 1, TILE):
    for x in range(0, w - TILE + 1, TILE):
        label_tile = label_img[y:y+TILE, x:x+TILE]
        clean_tile = clean_img[y:y+TILE, x:x+TILE]
        tile_overlay = overlay_u8[y:y+TILE, x:x+TILE]

        overlay_pixels = int((tile_overlay > 0).sum())
        has_yellow = int(overlay_pixels >= MIN_OVERLAY_PIXELS)

        if has_yellow: yellow1_count += 1
        else: yellow0_count += 1

        fname = f"{base}_x{x:05d}_y{y:05d}_yellow{has_yellow}.png"
        cv2.imwrite(str(train_path / fname), label_tile)
        cv2.imwrite(str(test_path / fname), clean_tile)
        tile_count += 1

print(f"Saved {tile_count} tiles to train/ and test/")
print(f"Label counts: yellow1={yellow1_count}, yellow0={yellow0_count}")


Loaded shapes: (4690, 6278, 3) (4690, 6278, 3)
Saved 432 tiles to train/ and test/
Label counts: yellow1=0, yellow0=432


In [12]:
import cv2
import numpy as np
from pathlib import Path

img_label = 'm05-label.png'
img_nolabel = 'm05-nolabel.png'

train_path = Path('train')
test_path = Path('test')
train_path.mkdir(exist_ok=True)
test_path.mkdir(exist_ok=True)

TILE = 256
MIN_YELLOW_PIXELS = 80   # tune if needed

def yellow_mask_bgr(img_bgr):
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    lower = np.array([18, 80, 80])
    upper = np.array([45, 255, 255])
    return cv2.inRange(hsv, lower, upper)

def iter_tiles_no_overlap(img, tile):
    h, w = img.shape[:2]
    for y in range(0, h - tile + 1, tile):
        for x in range(0, w - tile + 1, tile):
            yield x, y, img[y:y+tile, x:x+tile]

# Load images
label_img = cv2.imread(img_label)
clean_img = cv2.imread(img_nolabel)

if label_img is None:
    raise FileNotFoundError(f"Could not read {img_label}")
if clean_img is None:
    raise FileNotFoundError(f"Could not read {img_nolabel}")

if label_img.shape[:2] != clean_img.shape[:2]:
    raise ValueError("Label and no-label images must be same size / aligned")

yellow_mask = yellow_mask_bgr(label_img)

base = Path(img_nolabel).stem.replace('-nolabel', '')

train_count = 0
test_count = 0

# Tile across full grid
for x, y, label_tile in iter_tiles_no_overlap(label_img, TILE):
    clean_tile = clean_img[y:y+TILE, x:x+TILE]

    tile_mask = yellow_mask[y:y+TILE, x:x+TILE]
    yellow_pixels = int((tile_mask > 0).sum())
    has_yellow = int(yellow_pixels >= MIN_YELLOW_PIXELS)

    fname = f"{base}_x{x:05d}_y{y:05d}_yellow{has_yellow}.png"

    # Save labeled tiles to train/
    cv2.imwrite(str(train_path / fname), label_tile)
    train_count += 1

    # Save clean tiles to test/
    cv2.imwrite(str(test_path / fname), clean_tile)
    test_count += 1

print(f"Saved {train_count} tiles to {train_path}/ and {test_count} tiles to {test_path}/")

Saved 432 tiles to train/ and 432 tiles to test/
